# Circuit diagrams of the ten feature map families

This notebook draws the circuits of the paper **directly from the frozen `pqk_circuits.py`**, so every gate in a diagram is a gate that was simulated.
It needs only `numpy` and `matplotlib` (no LaTeX and no quantum SDK).

* The figures are drawn at the journal text width of 5.16 in. Drawing units are inches, so the font sizes below are the printed sizes.
* A two-qubit gate is one box spanning both of its wires, with its label written once.
* Section 1 sets what to draw. Change `N_QUBITS`, `GRAPH`, `LAYERS` or the two family groups and run all cells again.
* Section 3 writes `fig4_circuits_commuting.pdf` and `fig5_circuits_noncommuting.pdf` (Figs. 4 and 5 of the paper) and one file per family.

It imports `pqk_circuits.py` from the repository root (one folder up).

## 1. Settings

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle

sys.path.insert(0, str(Path.cwd().parent))   # pqk_circuits.py sits in the repository root
import pqk_circuits as pc

N_QUBITS = 4                       # qubits drawn
GRAPH = "chain"                    # "chain", "ring", "star", "complete" or a list of edges such as [(0, 1), (1, 3)]
LAYERS = 1                         # number of repeated layers
GROUP_A = ["zz_reupload", "havlicek_zz", "iqp", "cz_layers", "ms_xx"]       # commuting two-qubit gates (Fig. 4)
GROUP_B = ["xy", "heisenberg", "hea_cnot", "crx", "separable"]              # non-commuting and product-state control (Fig. 5)
OUT = Path("circuit_figures"); OUT.mkdir(exist_ok=True)

W = 5.16                           # figure width in inches (journal text width)
FONT = 8.5                         # gate labels in points
TITLE_FONT = 9
LW = 1.0                           # wire and box line width in points
plt.rcParams.update({"font.family": "DejaVu Sans", "mathtext.default": "it", "savefig.dpi": 300})

def graph_edges(name, n):
    if not isinstance(name, str):
        return [tuple(sorted(e)) for e in name]
    return {"chain": [(i, i + 1) for i in range(n - 1)],
            "ring": sorted({tuple(sorted((i, (i + 1) % n))) for i in range(n)}),
            "star": [(0, i) for i in range(1, n)],
            "complete": [(i, j) for i in range(n) for j in range(i + 1, n)]}[name]
EDGES = graph_edges(GRAPH, N_QUBITS)
print("edges:", EDGES)

## 2. Drawing code

In [ ]:
TITLE = {"zz_reupload": "ZZ re-uploading", "havlicek_zz": "Havlíček ZZ", "iqp": "IQP", "cz_layers": "CZ layers", "ms_xx": "MS XX",
         "xy": "XY", "heisenberg": "Heisenberg", "hea_cnot": "HEA CNOT", "crx": "CRX", "separable": "Separable control"}
FILL1, EDGE1 = "#e6effa", "#2a78d6"     # single-qubit gates
FILL2, EDGE2 = "#fce9e1", "#eb6834"     # data-dependent two-qubit gates
INK, WIRE = "#0b0b0b", "#52514e"
DY = 0.42          # distance between wires (in)
COLW = 0.90        # column width (in)
BW, BH = 0.82, 0.27  # box width and height of a single-qubit gate (in)
LEFT = 0.48        # space for the |0> labels (in)

def single_label(family, name, k):
    q = k + 1
    if name == "h":
        return r"$H$"
    if family == "havlicek_zz" and name == "rz":
        return rf"$R_z(2cx_{q})$"
    return rf"$R_{name[1]}(cx_{q})$"

def two_label(family, name, i, j):
    a, b = i + 1, j + 1
    if family == "havlicek_zz":
        return rf"$ZZ(\phi_{{{a}{b}}})$"
    sym = {"zz": "ZZ", "xx": "XX", "xy": "XY", "heis": r"\mathrm{Heis}"}[name]
    return rf"${sym}(c^2x_{a}x_{b})$"

def layout(gates, n):
    """Earliest column in which every wire the gate spans is free."""
    free = [0] * n; cols = []
    for kind, name, q, _ in gates:
        span = range(min(q), max(q) + 1)
        col = max(free[k] for k in span)
        for k in span: free[k] = col + 1
        cols.append(col)
    return cols, (max(free) if free else 0)

def box(ax, x, y0, y1, fill, edge, label):
    ax.add_patch(FancyBboxPatch((x - BW / 2, y0), BW, y1 - y0, boxstyle="round,pad=0,rounding_size=0.04",
                                fc=fill, ec=edge, lw=LW, zorder=2))
    ax.text(x, (y0 + y1) / 2, label, ha="center", va="center", fontsize=FONT, zorder=3)

def draw_circuit(ax, family, n=N_QUBITS, edges=EDGES, L=LAYERS, ncols=None, title=None):
    """Draws one family on an axis whose data units are inches."""
    gates = pc.build_circuit(family, np.zeros((1, n)), edges, L, 1.0)
    cols, ncol = layout(gates, n)
    ncols = ncols or ncol
    y = lambda k: (n - 1 - k) * DY
    x_end = LEFT + COLW * ncols + 0.1
    for k in range(n):
        ax.plot([LEFT - 0.05, x_end], [y(k), y(k)], color=WIRE, lw=LW, zorder=0)
        ax.text(LEFT - 0.08, y(k), rf"$|0\rangle_{k + 1}$", ha="right", va="center", fontsize=FONT)
    r_dot = 0.045; r_plus = 0.09
    for (kind, name, q, _), c in zip(gates, cols):
        x = LEFT + COLW * c + COLW / 2
        if kind == 1:
            k = q[0]; box(ax, x, y(k) - BH / 2, y(k) + BH / 2, FILL1, EDGE1, single_label(family, name, k)); continue
        i, j = q; top, bot = max(y(i), y(j)), min(y(i), y(j))
        if name in ("cnot", "cz", "crx"):
            ax.plot([x, x], [bot, top], color=INK, lw=LW, zorder=1)
            ax.add_patch(Circle((x, y(i)), r_dot, color=INK, zorder=3))
            if name == "cz":
                ax.add_patch(Circle((x, y(j)), r_dot, color=INK, zorder=3))
            elif name == "cnot":
                ax.add_patch(Circle((x, y(j)), r_plus, fc="white", ec=INK, lw=LW, zorder=3))
                ax.plot([x - r_plus, x + r_plus], [y(j), y(j)], color=INK, lw=LW, zorder=4)
                ax.plot([x, x], [y(j) - r_plus, y(j) + r_plus], color=INK, lw=LW, zorder=4)
            else:
                box(ax, x, y(j) - BH / 2, y(j) + BH / 2, FILL2, EDGE2, rf"$R_x(cx_{i + 1})$")
        else:
            box(ax, x, bot - BH / 2, top + BH / 2, FILL2, EDGE2, two_label(family, name, i, j))
    if L > 1:
        per = len(gates) // L
        for l in range(1, L):
            xb = LEFT + COLW * cols[l * per]; ax.plot([xb, xb], [-0.25, y(0) + 0.25], color="#a3a29d", ls=":", lw=0.8)
    ax.set_xlim(0, W); ax.set_ylim(-0.3, (n - 1) * DY + 0.3); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.text(0, (n - 1) * DY + 0.3, title, ha="left", va="bottom", fontsize=TITLE_FONT)
    return ncol

def panel_height(n=N_QUBITS):
    return (n - 1) * DY + 0.6 + 0.22   # wires, margins and title

def figure_for(families, name):
    h = panel_height(); H = h * len(families)
    fig = plt.figure(figsize=(W, H))
    for r, f in enumerate(families):
        ax = fig.add_axes([0, 1 - (r + 1) * h / H, 1, (h - 0.22) / H])
        draw_circuit(ax, f, title=f"({chr(97 + r)}) {TITLE[f]}")
    fig.savefig(OUT / f"{name}.pdf", bbox_inches="tight", pad_inches=0.02)
    fig.savefig(OUT / f"{name}.png", bbox_inches="tight", pad_inches=0.02)
    plt.show()

fig = plt.figure(figsize=(W, panel_height())); ax = fig.add_axes([0, 0, 1, (panel_height() - 0.22) / panel_height()])
draw_circuit(ax, "hea_cnot", title="HEA CNOT"); plt.show()

## 3. Figures 4 and 5 of the paper, and one file per family

In [ ]:
figure_for(GROUP_A, "fig4_circuits_commuting")
figure_for(GROUP_B, "fig5_circuits_noncommuting")
for f in GROUP_A + GROUP_B:
    fig = plt.figure(figsize=(W, panel_height())); ax = fig.add_axes([0, 0, 1, (panel_height() - 0.22) / panel_height()])
    draw_circuit(ax, f, title=TITLE[f])
    fig.savefig(OUT / f"circuit_{f}.pdf", bbox_inches="tight", pad_inches=0.02); plt.close(fig)
print("figures written to", OUT.resolve())